# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanEhssan/flyrank-internship-ml-01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

My provisional lane is **Refresh / Content Opportunity Scoring**. The unit of analysis is one pseudonymized content page represented by its trailing-90-day observations. I selected this lane because it combines observable visibility, movement, freshness, search-position, content, and engagement signals into a practical ranked review queue. The objective is not simply to train a model; it is to help an SEO specialist or content editor allocate limited review time to pages with the strongest evidence of potential opportunity. A transparent rules-based baseline will be built first, and machine learning will only be useful if it improves prioritization under honest validation.


In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("data/raw/content_refresh_anonymized.csv")
data_source = data_path if data_path.exists() else "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_source)

pd.Series({
    "pages": len(df),
    "clients": df["client_id"].nunique(),
    "columns": df.shape[1]
})

,0
pages,30000
clients,32
columns,44


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which content pages should an SEO specialist or content editor review first for a possible refresh, based on observed visibility, decline, freshness, position, CTR, content-depth, and engagement signals?

**Decision:** The project should improve the decision of which pages enter a limited weekly content-review queue and in what order.

**Output:** The intended output is a ranked page-level review queue containing an opportunity score, interpretable reason codes, and a suggested next action.

**Action:** A human reviewer can inspect the highest-ranked pages and decide whether to refresh the content, improve its title or metadata, strengthen intent alignment, consolidate it with related content, monitor it, or take no action.

**Cost of a false positive:** A page may be recommended despite having no meaningful opportunity. This wastes editorial time and could lead to unnecessary changes to content that was already performing appropriately.

**Cost of a false negative:** A genuinely declining or underperforming high-opportunity page may be ranked too low. The team may miss an opportunity to investigate it while it continues losing visibility, clicks, or engagement.

**Why data or ML may help:** The decision involves several interacting signals whose importance may vary across pages and clients. A simple rule may identify obvious cases, but it may not rank mixed or borderline cases effectively. Machine learning can be considered when those interactions are too complex for a stable hand-written rule. It must still be compared with a transparent baseline and retained only when it improves the top of the ranked queue on held-out data. The provisional evaluation metric will be Precision@K, where K represents the number of pages the editorial team can realistically review.

In [2]:
reviewable = (df["impressions_90d"] >= 100) | (df["sessions_90d"] >= 10)

pd.Series({
    "pages_with_minimum_activity": int(reviewable.sum()),
    "share_of_dataset_pct": round(reviewable.mean() * 100, 2)
})


,0
pages_with_minimum_activity,22827.00
share_of_dataset_pct,76.09


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 pseudonymized content pages from 32 clients**. Within the current snapshot, **16,726 pages, or 55.75%, recorded at least 500 impressions during the trailing 90-day period**. There are also **13,152 pages, or 43.84%, that are both trending down and have at least 100 impressions**, indicating a substantial set of visible pages that may deserve investigation. In addition, **9,759 pages, or 32.53%, have at least 500 impressions, an average position between 1 and 20, and a CTR below 0.5%**.

These measurements suggest that the dataset contains enough visible, declining, and potentially under-capturing pages to justify developing a ranked review process over the next seven weeks. They do not prove that these pages should be changed or that editing them would improve their performance.

In [3]:
visible_pages = df["impressions_90d"] >= 500

declining_with_demand = (
    df["trend_direction"].eq("down")
    & (df["impressions_90d"] >= 100)
)

low_ctr_visible = (
    visible_pages
    & df["avg_position"].between(1, 20)
    & (df["ctr"] < 0.5)
)

evidence = pd.DataFrame({
    "measure": [
        "Pages with at least 500 impressions",
        "Declining pages with at least 100 impressions",
        "Visible pages in positions 1-20 with CTR below 0.5%"
    ],
    "pages": [
        int(visible_pages.sum()),
        int(declining_with_demand.sum()),
        int(low_ctr_visible.sum())
    ],
    "share_of_dataset_pct": [
        round(visible_pages.mean() * 100, 2),
        round(declining_with_demand.mean() * 100, 2),
        round(low_ctr_visible.mean() * 100, 2)
    ]
})

evidence


,measure,pages,share_of_dataset_pct
0,Pages with at least 500 impressions,16726,55.75
1,Declining pages with at least 100 impressions,13152,43.84
2,Visible pages in positions 1-20 with CTR below...,9745,32.48


## 4. Careful words: what I can and can't claim

This project can describe **observed** and **measured** patterns within the anonymized dataset. If later validation is designed correctly, it may also show that one ranking method prioritizes observed page-level opportunities more effectively than a transparent baseline on held-out data. The resulting queue will remain a decision-support tool for human reviewers.

The project cannot prove that refreshing a page caused a change in search performance because the dataset is observational rather than a controlled intervention. It cannot predict or reverse-engineer Google's ranking algorithm, guarantee that a recommended page will recover, assume that every traffic decline represents content decay, or claim that results automatically generalize beyond the evaluated clients and time period. It will also not attempt to identify any real client, domain, URL, title, keyword, or query represented by the pseudonymized data.

In [4]:
prohibited_columns = {
    "client_name",
    "domain",
    "url",
    "title",
    "keyword",
    "query"
}

assert prohibited_columns.isdisjoint(df.columns)

"Public-safe starter fields confirmed"


'Public-safe starter fields confirmed'

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.